In [11]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from tqdm import tqdm



In [20]:
nltk.download('vader_lexicon')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\BINHPHAN\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\BINHPHAN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\BINHPHAN\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\BINHPHAN\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\BINHPHAN\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [13]:
sia = SentimentIntensityAnalyzer()
lemmatizer = WordNetLemmatizer()



In [14]:
print("====Loading data=======")
df = pd.read_csv('Reviews_withURL.csv')


====Loading data=======


In [15]:
df = df.dropna(subset=['Text'])


In [16]:
base_healthy_words= {'healthy', 'organic', 'whole grain', 'fiber', 'oat', 'quinoa', 'flaxseed', 'sugar free', 'no sugar', 'low sugar', 'stevia', 'unsweetened', 'nuts', 'almond', 'walnut', 'protein', 'omega', 'vegan', 'plant based', 'vegetarian', 'soy', 'natural', 'gluten free', 'non gmo', 'antioxidant', 'green tea', 'herbal', 'probiotic', 'kale', 'spinach', 'chia', 'avocado', 'honey', 'superfood'}


In [17]:
def is_healthy_nlp(text):
    if not isinstance(text, str):
        return False

    tokens = word_tokenize(text.lower())

    lemmatized_words = set(lemmatizer.lemmatize(word) for word in tokens)

    return len(base_healthy_words.intersection(lemmatized_words)) > 0


Step 1: Filtering Products related to Healthy Foods

In [ ]:
print('======Filtering========')
tqdm.pandas()
df['Is_Healthy'] = df['Text'].progress_apply(is_healthy_nlp)


Step 1: Filtering Healthy Foods


100%|██████████| 568454/568454 [07:09<00:00, 1322.44it/s]


In [22]:
healthy_df = df[df['Is_Healthy']].copy()


In [23]:
print(f"{len(healthy_df)} products related to healthy food were found")


111056 products related to healthy food were found


Step 2: Sentiment Analysis

In [24]:
def get_negative_sentiment(text):
    return sia.polarity_scores(text)['compound']


In [25]:
healthy_df['Sentiment_Score'] = healthy_df['Text'].progress_apply(get_negative_sentiment)


100%|██████████| 111056/111056 [02:01<00:00, 911.95it/s] 


In [26]:
negative_healthy_df = healthy_df[(healthy_df['Sentiment_Score'] < -0.05) & (healthy_df['Score'] <= 3)]


In [28]:
print(f"{len(negative_healthy_df)} products received negative reviews.")


4771 products received negative reviews.


In [31]:
if not negative_healthy_df.empty:
    # Đếm số lượng đánh giá tiêu cực theo từng sản phẩm
    product_counts = negative_healthy_df.groupby(['ProductId', 'ProductURL']).size().reset_index(name='Negative_NLP_Count')
    
    # Sắp xếp để lấy sản phẩm bị chê nhiều nhất
    top_negative = product_counts.sort_values(by='Negative_NLP_Count', ascending=False)
    worst_product = top_negative.iloc[0]
    
    print("\n" + "="*50)
    print("The healthy food with the most negative feedbacks:")
    print(f"ProductId: {worst_product['ProductId']}")
    print(f"URL:         {worst_product['ProductURL']}")
    print(f"Number of negative reviews: {worst_product['Negative_NLP_Count']}")
    print("="*50)
    
    # In thử 3 review tiêu cực nhất của sản phẩm này để kiểm chứng
    print("\nSome negative reviews:")
    sample_reviews = negative_healthy_df[negative_healthy_df['ProductId'] == worst_product['ProductId']].sort_values(by='Sentiment_Score').head(3)
    for idx, row in sample_reviews.iterrows():
        print(f"- Sentiment Score VADER: {row['Sentiment_Score']} | Stars (Score): {row['Score']}")
        print(f"  Content: {row['Summary']} - {row['Text'][:150]}...\n")
else:
    print("No data available.")



The healthy food with the most negative feedbacks:
ProductId: B001VJ0B0I
URL:         https://www.amazon.com/dp/B001VJ0B0I
Number of negative reviews: 18

Some negative reviews:
- Sentiment Score VADER: -0.9651 | Stars (Score): 1
  Content: Good food for those who hate their dogs - But who hates their dog? We sure don't, so we try to give her what she wants and needs, and neither of those things are in this food.<br /><br />The n...

- Sentiment Score VADER: -0.9071 | Stars (Score): 3
  Content: Not my dogs's favorite - I consider Purina a reputable brand and my golden retriever really likes Purina Dog Chow, so this seemed like a healthy alternative. My dog ate the fo...

- Sentiment Score VADER: -0.8995 | Stars (Score): 2
  Content: Are all the pretty colors really necessary?  Or Healthy? - This is dog food and therefore, I did not consume it myself -- however, based on the colors and shapes of the kibbles, I think Beneful assumes that I ...

